---
# <div align="center"><font color='blue'> COSC 2779/2972 | Deep Learning </font></div>
## <div align="center"><font color='blue'> Week 5 Lab: CNN Architectures, Custom Datasets & Transfer Learning</font></div>
---

## 🎯 Learning Objectives

By the end of this lab, you should be able to:

- Load and structure a custom image dataset using `ImageFolder` and `DataLoader`
- Build CNN pipelines to solve a real vision problem
- Understand how architecture choices affect performance
- Apply transfer learning
- Decide when transfer learning is appropriate and when it isn't
- Use standard experiment tracking tools

**Prerequisites:** Week 3 (optimisation), Week 4 (CNNs & data pipelines)

**Duration:** ~2–3 hours

---
## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Training Utilities

We reuse the same training and evaluation functions from Weeks 3–4.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item() * y.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total


def run_experiment(model, optimizer, criterion, train_loader, val_loader,
                   epochs=10, name='Model'):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  [{name}] Epoch {epoch+1}/{epochs} — '
                  f'Train: {train_loss:.4f} / {train_acc:.4f} | '
                  f'Val: {val_loss:.4f} / {val_acc:.4f}')
    return history


def plot_experiments(results, show_train=True, show_val=True):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, hist in results.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        if show_train:
            axes[0].plot(epochs, hist['train_loss'], label=f'{name} (Train)', marker='o', markersize=3)
            axes[1].plot(epochs, hist['train_acc'], label=f'{name} (Train)', marker='o', markersize=3)
        if show_val:
            axes[0].plot(epochs, hist['val_loss'], label=f'{name} (Val)', linestyle='--', marker='x', markersize=3)
            axes[1].plot(epochs, hist['val_acc'], label=f'{name} (Val)', linestyle='--', marker='x', markersize=3)
    for ax, ylabel, title in zip(axes, ['Loss', 'Accuracy'], ['Loss', 'Accuracy']):
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

---
## Part 1: Custom Dataset Setup

So far, we've used built-in datasets (MNIST, CIFAR-10). In practice, you will often work with **custom datasets** that come as folders of images. PyTorch's `ImageFolder` makes this straightforward.

### Dataset: Cats vs Dogs (small subset)

We'll use a subset of the [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/), pre-organised into the folder structure PyTorch expects.

The expected folder structure is:
```
data/
  train/
    cats/
      cat_001.jpg
      cat_002.jpg
      ...
    dogs/
      dog_001.jpg
      dog_002.jpg
      ...
  val/
    cats/
      ...
    dogs/
      ...
```

`ImageFolder` automatically assigns class labels based on subfolder names.

### Downloading and Preparing the Data

The following code downloads the Oxford-IIIT Pet Dataset and organises it into a train/val split with the folder structure shown above. Run this once.

In [ ]:
import shutil
from pathlib import Path

def setup_pet_dataset(data_dir='data', train_ratio=0.7, val_ratio=0.15, max_per_class=500):
    """
    Download Oxford-IIIT Pet Dataset and organise into train/val/test folders.
    Uses a subset (max_per_class) for faster training.

    Split: 70% train, 15% val, 15% test (by default)
    """
    data_path = Path(data_dir)

    # Skip if already prepared
    if all((data_path / split / 'cat').exists() for split in ['train', 'val', 'test']):
        n_train = sum(1 for _ in (data_path / 'train').rglob('*.jpg'))
        n_val = sum(1 for _ in (data_path / 'val').rglob('*.jpg'))
        n_test = sum(1 for _ in (data_path / 'test').rglob('*.jpg'))
        if n_train > 0:
            print(f'Dataset already prepared: {n_train} train, {n_val} val, {n_test} test images')
            return

    # Download
    print('Downloading Oxford-IIIT Pet Dataset...')
    pet_dataset = datasets.OxfordIIITPet(
        root=data_dir, split='trainval', download=True
    )

    # Filenames starting with uppercase = cat breeds, lowercase = dog breeds
    images_dir = Path(data_dir) / 'oxford-iiit-pet' / 'images'

    cat_images, dog_images = [], []
    for img_file in sorted(images_dir.glob('*.jpg')):
        breed_name = '_'.join(img_file.stem.split('_')[:-1])
        if breed_name[0].isupper():
            cat_images.append(img_file)
        else:
            dog_images.append(img_file)

    print(f'Found {len(cat_images)} cat images, {len(dog_images)} dog images')

    # Subsample and shuffle
    np.random.seed(42)
    np.random.shuffle(cat_images)
    np.random.shuffle(dog_images)
    cat_images = cat_images[:max_per_class]
    dog_images = dog_images[:max_per_class]

    # Create folder structure
    for split in ['train', 'val', 'test']:
        for cls in ['cat', 'dog']:
            (data_path / split / cls).mkdir(parents=True, exist_ok=True)

    # Split and copy
    for class_name, images in [('cat', cat_images), ('dog', dog_images)]:
        n = len(images)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        for i, img_path in enumerate(images):
            if i < train_end:
                split = 'train'
            elif i < val_end:
                split = 'val'
            else:
                split = 'test'
            dest = data_path / split / class_name / f'{class_name}_{i:04d}.jpg'
            shutil.copy2(img_path, dest)

    n_train = sum(1 for _ in (data_path / 'train').rglob('*.jpg'))
    n_val = sum(1 for _ in (data_path / 'val').rglob('*.jpg'))
    n_test = sum(1 for _ in (data_path / 'test').rglob('*.jpg'))
    print(f'Dataset prepared: {n_train} train, {n_val} val, {n_test} test images')

setup_pet_dataset()

### Loading the Dataset with `ImageFolder`

`ImageFolder` reads the folder structure and automatically assigns integer labels to each class (alphabetically). It applies the specified transforms to each image on-the-fly.

Since images in the pet dataset vary in size, we **resize** all images to a fixed resolution. We'll use 128×128 for our custom CNNs and 224×224 when using pretrained models (which were trained on ImageNet at that resolution).

In [ ]:
# Image size for our custom CNNs
IMG_SIZE = 128

# Basic transforms (no augmentation yet)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder('data/train', transform=train_transform)
val_dataset = datasets.ImageFolder('data/val', transform=val_transform)
test_dataset = datasets.ImageFolder('data/test', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


class_names = train_dataset.classes
num_classes = len(class_names)

print(f'Classes: {class_names}')
print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Test samples:       {len(test_dataset)}')

---
### Dataset Exploration

Always inspect your data before training. This is especially important with custom datasets where issues like corrupted images, mislabelled samples, or inconsistent preprocessing can waste hours of training time.

In [ ]:
# Visualise sample images
images, labels = next(iter(train_loader))

print(f'Batch shape: {images.shape}')
print(f'Pixel range: [{images.min():.3f}, {images.max():.3f}]')

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(class_names[labels[i].item()], fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Check class distribution
from collections import Counter

train_labels = [label for _, label in train_dataset]
val_labels = [label for _, label in val_dataset]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, split_labels, title in zip(axes, [train_labels, val_labels], ['Training', 'Validation']):
    counts = Counter(split_labels)
    ax.bar([class_names[k] for k in sorted(counts)],
           [counts[k] for k in sorted(counts)],
           color='steelblue', edgecolor='white')
    ax.set_title(f'{title} Set')
    ax.set_ylabel('Count')

plt.suptitle('Class Distribution', fontsize=12)
plt.tight_layout()
plt.show()

### 🤔 Questions

1. Is the dataset balanced? If not, what strategies could you use to handle the imbalance?
2. What do you notice about the images? Are they clean and consistent, or noisy and varied?
3. Why do we resize all images to the same dimensions? What are the trade-offs of choosing a larger vs smaller image size?
4. Why are the training and validation transforms the same for now? When would we want them to differ?

*Double-click here to write your answers for Part 0...*

---
## Part 2: Baseline CNN from Scratch

Let's start with a CNN similar to the one from Week 4, adapted for our new image size and number of classes.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            # Input: (3, 128, 128)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),              # -> (32, 64, 64)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),              # -> (64, 32, 32)

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),              # -> (128, 16, 16)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),      # -> (128, 1, 1) — global average pooling
            nn.Flatten(),                 # -> (128,)
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

baseline_cnn = BaselineCNN(num_classes=num_classes)
n_params = sum(p.numel() for p in baseline_cnn.parameters())
print(f'Baseline CNN parameters: {n_params:,}')
print(baseline_cnn)

👉 **Note:** We use `AdaptiveAvgPool2d(1)` (global average pooling) instead of flattening the entire feature map. This reduces `128 × 16 × 16 = 32,768` features to just `128`, dramatically reducing the number of parameters in the classifier and acting as a mild regulariser.

In [ ]:
torch.manual_seed(0)
baseline_cnn = BaselineCNN(num_classes=num_classes).to(device)
optimizer = optim.Adam(baseline_cnn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

results = {}
results['Baseline CNN'] = run_experiment(
    baseline_cnn, optimizer, criterion, train_loader, val_loader,
    epochs=15, name='Baseline'
)

plot_experiments(results)

### 🤔 Questions

1. Does the model converge? What does the loss curve look like?
2. Is there a gap between training and validation accuracy? What does this indicate?
3. What is the baseline validation accuracy? Is this reasonable for a binary classification task? (Remember: random guessing would give ~50%)
4. What accuracy would you aim for as a target?

*Double-click here to write your answers for Part 1...*

---
## Part 3: Architecture Matters

Let's explore how different architecture choices affect performance. We'll compare:
- A **shallower** CNN (fewer layers)
- A **deeper** CNN (more layers)
- The baseline from Part 1

🤔 **Think:** Before running the experiments, predict which architecture will perform best and why.

In [ ]:
class ShallowCNN(nn.Module):
    """A minimal CNN — just one conv block."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class DeeperCNN(nn.Module):
    """A deeper CNN — five conv blocks."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
arch_results = {}
arch_results['Baseline (3 blocks)'] = results['Baseline CNN']
criterion = nn.CrossEntropyLoss()

for name, ModelClass in [('Shallow (1 block)', ShallowCNN), ('Deeper (5 blocks)', DeeperCNN)]:
    torch.manual_seed(0)
    model = ModelClass(num_classes=num_classes).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'{name}: {n_params:,} parameters')
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    arch_results[name] = run_experiment(
        model, optimizer, criterion, train_loader, val_loader,
        epochs=15, name=name
    )
    print()

plot_experiments(arch_results)

### 🤔 Questions

1. Does deeper always mean better? What do you observe?
2. Which architecture trains fastest (reaches good accuracy in fewest epochs)?
3. Which generalises best (smallest train/val gap)?
4. How do the parameter counts compare? Is there a relationship between parameter count and overfitting?
5. 🤔 **Think:** If you had 10× more training data, would the deeper model benefit more than the shallower one? Why?

*Double-click here to write your answers for Part 2...*

---
## Part 4: Transfer Learning

Training a CNN from scratch requires a lot of data. With only ~800 training images, our custom CNNs are likely to overfit. **Transfer learning** addresses this by starting from a model pretrained on a large dataset (ImageNet, 1.2M images, 1000 classes).

### The Transfer Learning Recipe

Transfer learning is done in **two sequential steps**, with second step being optional:

1. **Linear probing (frozen backbone):** Freeze all pretrained layers and train only a new classification head. The pretrained network acts as a fixed feature extractor. This gives the randomly initialised head a chance to learn reasonable weights before we touch the backbone.

2. **Fine-tuning (unfreeze backbone):** Starting from the model trained in step 1, unfreeze some or all of the backbone layers and continue training with a **reduced learning rate**. The lower learning rate is critical — the pretrained features are already good, and we want to gently adapt them rather than destroy them with large gradient updates.

⚠️ **Why must these steps be sequential?** The new classification head starts with random weights. If we unfreeze the backbone immediately, the large random gradients from the untrained head flow backward and corrupt the pretrained features. Training the head first ensures that when we unfreeze the backbone, the gradients flowing back are meaningful rather than noise.

We'll use **ResNet-18** — a well-known architecture with ~11M parameters, pretrained on ImageNet.

👉 **Important:** Pretrained models expect inputs normalised with ImageNet statistics and sized 224×224. We need to update our transforms.

---

### Setting up Transforms for Pretrained Models

In [ ]:
# ImageNet normalisation statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

pretrained_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

pretrained_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Reload datasets with pretrained transforms
pretrained_train_dataset = datasets.ImageFolder('data/train', transform=pretrained_train_transform)
pretrained_val_dataset = datasets.ImageFolder('data/val', transform=pretrained_val_transform)
pretrained_test_dataset = datasets.ImageFolder('data/test', transform=pretrained_val_transform)


pretrained_train_loader = DataLoader(pretrained_train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
pretrained_val_loader = DataLoader(pretrained_val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
pretrained_test_loader = DataLoader(pretrained_test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


print(f'Train: {len(pretrained_train_dataset)}, Val: {len(pretrained_val_dataset)}, , Test: {len(pretrained_test_dataset)}')

### Preparing the Pretrained Model

We load ResNet-18 with ImageNet weights and replace its final classification layer to match our number of classes. A helper function lets us freeze or unfreeze the backbone.

In [ ]:
def create_resnet18(num_classes):
    """Load a pretrained ResNet-18 and replace the final layer."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def freeze_backbone(model):
    """Freeze all layers except the final classification head."""
    for name, param in model.named_parameters():
        if 'fc' not in name:  # freeze everything except the fc layer
            param.requires_grad = False


def unfreeze_backbone(model):
    """Unfreeze all layers for fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True


def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {trainable:,} / {total:,} parameters ({trainable/total*100:.1f}%)')

### Step 1 — Linear Probing (Frozen Backbone)

We freeze the entire pretrained backbone and train only the new classification head. This is fast (very few trainable parameters) and gives the head a good initialisation before we touch the backbone.

In [ ]:
torch.manual_seed(0)
model_tl = create_resnet18(num_classes).to(device)
freeze_backbone(model_tl)

print('Step 1: Linear Probing')
count_parameters(model_tl)

# Only optimise parameters that require gradients (the head)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_tl.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

tl_results = {}
tl_results['Step 1: Linear Probing'] = run_experiment(
    model_tl, optimizer, criterion, pretrained_train_loader, pretrained_val_loader,
    epochs=5, name='Linear Probing'
)

plot_experiments(tl_results)

### Step 2 — Fine-Tuning (Unfreeze Backbone)

Now we unfreeze the backbone and **continue training the same model** from step 1. Two critical details:

1. **Same model:** We do NOT create a new ResNet — we continue from `model_tl` which already has a trained head.
2. **Reduced learning rate:** We use a much smaller learning rate (typically 10–100× smaller than step 1) to gently adapt the pretrained features without destroying them.

👉 **Note:** We also create a **new optimiser** after unfreezing. This is necessary because the old optimiser only knew about the head parameters. The new optimiser needs to see all parameters.

In [ ]:
# Unfreeze the backbone
unfreeze_backbone(model_tl)

print('Step 2: Fine-Tuning (continuing from linear probing)')
count_parameters(model_tl)

# New optimiser with ALL parameters and a SMALLER learning rate
optimizer = optim.Adam(model_tl.parameters(), lr=1e-5)

tl_results['Step 2: Fine-Tuning'] = run_experiment(
    model_tl, optimizer, criterion, pretrained_train_loader, pretrained_val_loader,
    epochs=10, name='Fine-Tuning'
)

plot_experiments(tl_results)

### Comparing with Training from Scratch

Let's see how much transfer learning helps by comparing against a randomly initialised ResNet-18 trained from scratch.

In [ ]:
torch.manual_seed(0)
model_scratch = create_resnet18(num_classes).to(device)

# Reinitialise all weights (destroy pretrained weights)
for m in model_scratch.modules():
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3)

comparison_results = {}
comparison_results['Transfer Learning'] = tl_results['Step 2: Fine-Tuning']
comparison_results['From Scratch'] = run_experiment(
    model_scratch, optimizer, criterion, pretrained_train_loader, pretrained_val_loader,
    epochs=15, name='From Scratch'
)

plot_experiments(comparison_results)

### 🤔 Questions

1. How does the linear probing accuracy compare to the baseline CNN from Part 1? What does this tell you about the quality of the pretrained features?
2. Does fine-tuning (step 2) improve on linear probing (step 1)? By how much?
3. Why do we use `lr=1e-3` for linear probing but `lr=1e-5` for fine-tuning?
4. How does the transfer learning model compare to ResNet-18 trained from scratch? Why is the gap so large with only ~700 training images?
5. What percentage of the model's parameters are trainable during linear probing? Why is this so efficient?
6. 🤔 **Think:** Would transfer learning from ImageNet help if your target task was classifying microscope images of cells? What about X-ray images? At what point does the domain gap become too large?

*Double-click here to write your answers for Part 3...*

---
## Part 5: Full Model Comparison

Let's bring together all the models we've trained and compare them side by side.

In [ ]:
# Collect all results
all_results = {}
all_results['Baseline CNN'] = results['Baseline CNN']
all_results['Linear Probing'] = tl_results['Step 1: Linear Probing']
all_results['Fine-Tuning'] = tl_results['Step 2: Fine-Tuning']
all_results['From Scratch'] = comparison_results['From Scratch']

plot_experiments(all_results)

# Summary table
print(f'{"Model":<25} {"Best Val Acc":<15} {"Final Train Acc":<15}')
print('-' * 55)
for name, hist in all_results.items():
    best_val = max(hist['val_acc'])
    final_train = hist['train_acc'][-1]
    print(f'{name:<25} {best_val:<15.4f} {final_train:<15.4f}')

### 🤔 Questions

1. Which model performs best overall?
2. Which model is most **parameter-efficient** (best accuracy per parameter)?
3. If you were deploying this in a mobile application where model size matters, which would you choose?
4. If you had unlimited compute but only this training data, which approach would you use?

*Double-click here to write your answers for Part 4...*

---
## Part 6: Model Analysis

Let's analyse the best model's predictions in detail.

### Misclassification Analysis

In [ ]:
# Use the fine-tuned model for analysis
best_model = model_tl

# We need un-normalised images for visualisation
vis_val_dataset = datasets.ImageFolder('data/val', transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]))
vis_val_loader = DataLoader(vis_val_dataset, batch_size=64, shuffle=False)

# Also get normalised predictions
best_model.eval()
all_preds, all_labels = [], []

with torch.inference_mode():
    for x, y in pretrained_val_loader:
        x = x.to(device)
        outputs = best_model(x)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Find misclassified indices
errors = np.where(all_preds != all_labels)[0]
print(f'Misclassified: {len(errors)} / {len(all_labels)} ({len(errors)/len(all_labels)*100:.1f}%)')

In [ ]:
# Display misclassified images
n_show = min(12, len(errors))

if n_show > 0:
    fig, axes = plt.subplots(2, min(6, n_show), figsize=(16, 6))
    for i, ax in enumerate(axes.flat):
        if i >= n_show:
            ax.axis('off')
            continue
        idx = errors[i]
        img, _ = vis_val_dataset[idx]
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.set_title(f'True: {class_names[all_labels[idx]]}\nPred: {class_names[all_preds[idx]]}', fontsize=9)
        ax.axis('off')
    plt.suptitle('Misclassified Images', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No misclassified images!')

### Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

print(classification_report(all_labels, all_preds, target_names=class_names))

### 🤔 Questions

1. What types of images does the model get wrong? Are the errors reasonable?
2. Is the model better at recognising cats or dogs? Check the per-class metrics.
3. Looking at the misclassified images, can you identify any common patterns (e.g., unusual poses, occluded faces, ambiguous images)?

*Double-click here to write your answers for Part 5...*

---
## Part 7: Experiment Tracking with Weights & Biases

### Why Experiment Tracking?

So far, we've been tracking experiments manually — printing metrics, saving plots, and comparing results by eye. This works for a few experiments, but quickly becomes unmanageable when you're running dozens of configurations.

**Weights & Biases (W&B)** is an experiment tracking platform that automatically logs:
- Training and validation metrics (loss, accuracy, etc.) — with interactive live charts
- Hyperparameters (learning rate, batch size, architecture, etc.)
- System metrics (GPU utilisation, memory usage)
- Model checkpoints

All experiments are stored in a centralised dashboard, making it easy to compare runs, share results with collaborators, and reproduce experiments.

> 👉 W&B is free for academic use. Sign up at [wandb.ai](https://wandb.ai).

### Setup

First, install W&B and log in. You only need to do this once per Colab session.

In [ ]:
!pip install -q wandb

import wandb

# Log in — this will prompt you for an API key on first use.
# Get your key from: https://wandb.ai/authorize
wandb.login()

### Integrating W&B into the Training Loop

The integration requires just **three changes** to our existing code:

1. **`wandb.init()`** — start a new run and log hyperparameters
2. **`wandb.log()`** — log metrics each epoch
3. **`wandb.finish()`** — end the run

Let's create a W&B-enabled version of our `run_experiment` function:

In [ ]:
def run_experiment_wandb(model, optimizer, criterion, train_loader, val_loader,
                         epochs=10, name='Model', config=None):
    """
    Run a training experiment with Weights & Biases tracking.

    Args:
        config: dict of hyperparameters to log (lr, batch_size, etc.)
    """
    # 1. Initialise a W&B run
    run = wandb.init(
        project='week5-cnn-lab',    # all runs in this lab go to the same project
        name=name,                   # a human-readable name for this run
        config=config or {},         # hyperparameters to log
        reinit=True,                 # allow multiple runs in one notebook
    )

    # Optionally watch the model (logs gradients and parameter histograms)
    wandb.watch(model, criterion, log='all', log_freq=10)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # 2. Log metrics to W&B
        wandb.log({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
        })

        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  [{name}] Epoch {epoch+1}/{epochs} — '
                  f'Train: {train_loss:.4f} / {train_acc:.4f} | '
                  f'Val: {val_loss:.4f} / {val_acc:.4f}')

    # 3. Finish the run
    wandb.finish()

    return history

### Running Experiments with W&B

Let's re-run the above comparison, but now with full experiment tracking. Each run will appear in your W&B dashboard automatically.

In [ ]:
criterion = nn.CrossEntropyLoss()

# --- Step 1: Linear Probing ---
torch.manual_seed(0)
model_wb = create_resnet18(num_classes).to(device)
freeze_backbone(model_wb)

optimizer_wb = optim.Adam(
    filter(lambda p: p.requires_grad, model_wb.parameters()), lr=1e-3
)

wandb_results = {}
wandb_results['Linear Probing'] = run_experiment_wandb(
    model_wb, optimizer_wb, criterion,
    pretrained_train_loader, pretrained_val_loader,
    epochs=5, name='Linear-Probing',
    config={
        'model': 'resnet18',
        'strategy': 'linear_probing',
        'lr': 1e-3,
        'epochs': 5,
        'batch_size': 32,
        'augmentation': False,
        'dataset': 'oxford-pets-catdog',
    }
)
print()

# --- Step 2: Fine-Tuning (continue from the same model) ---
unfreeze_backbone(model_wb)

optimizer_wb = optim.Adam(model_wb.parameters(), lr=1e-5)

wandb_results['Fine-Tuning'] = run_experiment_wandb(
    model_wb, optimizer_wb, criterion,
    pretrained_train_loader, pretrained_val_loader,
    epochs=10, name='Fine-Tuning',
    config={
        'model': 'resnet18',
        'strategy': 'fine_tuning',
        'lr': 1e-5,
        'epochs': 10,
        'batch_size': 32,
        'augmentation': False,
        'dataset': 'oxford-pets-catdog',
    }
)
print()

### Viewing Your Results

In [ ]:
# Print the link to your W&B project dashboard
print('\n' + '='*60)
print('Open your W&B dashboard to compare all experiments:')
print(f'  https://wandb.ai/{wandb.api.default_entity}/week5-cnn-lab')
print('='*60)

### What to Explore in the W&B Dashboard

Once you've run the experiments above, open the W&B project page and try:

1. **Compare loss curves:** Select all three runs and view the `val_loss` and `train_loss` charts side by side. Which converges fastest?
2. **Compare hyperparameters:** Click the "Table" view to see all hyperparameters for each run in a table. This makes it easy to spot what's different between runs.
3. **System metrics:** Check the "System" tab to see GPU utilisation. Was the GPU fully utilised, or was data loading the bottleneck?
4. **Parallel coordinates:** In the runs table, use the "Add panel" → "Parallel coordinates" to visualise the relationship between hyperparameters and final validation accuracy.

### Saving a Model Checkpoint as a W&B Artifact

You can log model checkpoints as W&B **Artifacts** for reproducibility and version control. This ensures you can always reload the exact model that produced a given result.

In [ ]:
# Save the best model as a W&B artifact
torch.save(model_wb.state_dict(), 'best_model.pth')

run = wandb.init(project='week5-cnn-lab', name='save-best-model', reinit=True)

artifact = wandb.Artifact('best-catdog-model', type='model')
artifact.add_file('best_model.pth')
wandb.log_artifact(artifact)

wandb.finish()
print('Model saved as W&B artifact — viewable in the Artifacts tab of your project.')

### 🤔 Questions

1. What advantages does W&B provide over manually plotting training curves in the notebook?
2. How does logging hyperparameters alongside metrics help when you have many experiments?
3. In a team setting, why is centralised experiment tracking important?
4. 🤔 **Think:** You've been running experiments for a week and have 50 runs in your project. How would you use the W&B dashboard to find the best-performing configuration?

*Double-click here to write your answers for Part 7...*

---
## Part 8: Mini Project

Improve performance as much as possible.

### What you can try:
- Different pretrained architectures (`resnet34`, `resnet50`, `efficientnet_b0`)
- Different augmentation strategies
- Learning rate scheduling (`CosineAnnealingLR`, `ReduceLROnPlateau`)
- Unfreezing only the last few layers instead of the entire backbone
- Training for more epochs

### Constraints:
- Use only the provided training data
- Keep total training time under 10 minutes

In [ ]:
# Your code here — build your best model
# ...

# Example: Try a different backbone
# model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
# model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)

### Final Test Set Evaluation

Use the test set as your final evaluation.

> 🚨 **In your assignments:** Always maintain a separate test set that is never used during model development.

---
## Part 9: Reflection

Answer the following questions in your own words:

1. When would you **NOT** use transfer learning? Give at least two scenarios.
2. In this lab, what mattered more for performance — the model architecture or the training data size? How do you know?
3. What was the single biggest improvement you made? Why was it so effective?
4. If you were given 10× more labelled cat/dog images, what would change about your approach?
5. What are the trade-offs between feature extraction and fine-tuning? When would you prefer each?
6. How does experiment tracking with a tool like W&B change your workflow compared to manually tracking results? Would you use it in your assignments?

*Double-click here to write your reflection answers...*

---

## 🔑 Key Takeaway

Transfer learning is one of the most powerful tools in deep learning. A model trained on millions of images learns general-purpose features (edges, textures, shapes) that transfer remarkably well to new tasks — even with very little data. Knowing when to freeze, when to fine-tune, and how to set up your data pipeline correctly is essential for real-world deep learning projects.